# TC-PhysXAI: Physics-Grounded XAI for Tropical Cyclone Track Prediction

**Author:** Nguyen Hong Gam  
**Paper:** TC-XFlow + TC-PhysXAI (ESWA 2025 + Extension)  
**Environment:** Kaggle GPU (P100/T4)

---

## Overview

| Module | Contribution | Description |
|--------|-------------|-------------|
| Module 1 | **C1** | Temporal attribution via climatological perturbation (ΔCRPS per 6h window) |
| Module 2 | **C2** | Integrated Gradients on FNO3D + physical object detection |
| Module 2 | **C3** | H-mean-F fidelity metric = 0.4·P + 0.4·R + 0.2·U |
| Module 3 | **M3** | DTW K-medoids scenario clustering + latent analogue retrieval |
| C4 | **C4** | FM integration-path attribution |
| C5 | **C5** | Variance decomposition: V_total = V_noise + V_context |
| C6 | **C6** | FM counterfactual via projected gradient on context space |
| **NEW** | **C7** | Segment-level temporal XAI + AOPC faithfulness |
| **NEW** | **C8** | Track Feature Attribution: speed/curvature/heading → ERA5 drivers |
| **NEW** | **E9** | So sánh với XAI4Traj (baseline của thầy) trên nhiều model, giao thức R/P/H |

> Notebook này chỉ chứa lệnh chạy. Toàn bộ code nằm trong GitHub repo.

---

## 1. Setup

1. Notebook settings → **Internet: On**, **GPU: On**.
2. Add Data: `tc-ofm` (kaggle1234uitvn) và `new-checkpoint` (gmnguynhng).
3. Chạy lần lượt 1.1 → 1.3; chỉ chạy tiếp khi 1.3 báo `ALL OK`.

Code: https://github.com/Hana-ai-18/ixpolainnoboai

In [ ]:
# ── 1.1 Clone repo GitHub (chứa cả TC-PhysXAI và Tropical_Typhoon_Architecture) ──
# Cần bật Internet: Notebook settings → Internet → On
!rm -rf /tmp/ixpolainnoboai
!git clone --depth 1 https://github.com/Hana-ai-18/ixpolainnoboai /tmp/ixpolainnoboai
!ls /tmp/ixpolainnoboai

In [ ]:
%%bash
# ── 1.2 Cài đặt vào /kaggle/working ──
set -e
rm -rf /kaggle/working/TC-PhysXAI /kaggle/working/Tropical_Typhoon_Architecture
cp -r /tmp/ixpolainnoboai/TC-PhysXAI /kaggle/working/TC-PhysXAI
cp -r /tmp/ixpolainnoboai/Tropical_Typhoon_Architecture /kaggle/working/Tropical_Typhoon_Architecture
mkdir -p /kaggle/working/TC-PhysXAI/results
ls /kaggle/working/TC-PhysXAI /kaggle/working/Tropical_Typhoon_Architecture/Model

In [ ]:
# ── 1.3 Kiểm tra môi trường (PhysXAI, Model TC-XFlow, checkpoint, dataset, GPU) ──
!python /kaggle/working/TC-PhysXAI/Scripts/check_env.py

## 2. Run Experiments

- **Option A (Recommended):** Multi-seed — E1 chạy 3 seeds, E2–E8 chạy seed0
- **Option B:** Single-seed — tất cả chạy seed0

> Thời gian T4: Option A ≈ 45–90 phút | Option B ≈ 30–60 phút

In [ ]:
# ── Option A (RECOMMENDED): Multi-Seed E1–E8 ────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/run_multiseed.py \
    --checkpoints \
        /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
        /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed1_new_ver.pth \
        /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed2_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 20 --K 30 \
    --experiments E1 E2 E3 E4 E5 E6 E7 E8

In [ ]:
# ── Option B: Single-Seed E1–E8 ─────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/run_all_experiments.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 20 --K 30 \
    --experiments E1 E2 E3 E4 E5 E6 E7 E8

## 3. Chạy từng Experiment

In [ ]:
# ── E1
!python /kaggle/working/TC-PhysXAI/Experiments/E1_comparison_baselines.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 30 --gpu 0

In [ ]:
# ── E2
!python /kaggle/working/TC-PhysXAI/Experiments/E2_recurvature_attribution.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 30 --gpu 0

In [ ]:
# ── E3
!python /kaggle/working/TC-PhysXAI/Experiments/E3_rapid_intensification.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 15 --gpu 0

In [ ]:
# ── E4
!python /kaggle/working/TC-PhysXAI/Experiments/E4_multimodal_consistency.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 20 --gpu 0

In [ ]:
# ── E5
!python /kaggle/working/TC-PhysXAI/Experiments/E5_fidelity_benchmark.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 15 --gpu 0

In [ ]:
# ── E6
!python /kaggle/working/TC-PhysXAI/Experiments/E6_counterfactual_scenarios.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 15 --gpu 0

In [ ]:
# ── E7
!python /kaggle/working/TC-PhysXAI/Experiments/E7_adaptive_segment_xai.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 20 --gpu 0

In [ ]:
# ── E8
!python /kaggle/working/TC-PhysXAI/Experiments/E8_track_feature_attribution.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 10 --gpu 0

## 4. Full Pipeline

In [ ]:
# ── Full: 20 storms ─────────────────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Scripts/run_physxai.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 20 --K 30 --run_experiments

In [ ]:
# ── Quick test: 1 storm ─────────────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Scripts/run_physxai.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 1 --K 10

## 5. Xem kết quả

In [ ]:
!find /kaggle/working/TC-PhysXAI/results -type f | sort

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/kaggle/working/TC-PhysXAI/results/**/*.png', recursive=True))[:6]:
    print(p); display(Image(p))

In [ ]:
!cat /kaggle/working/TC-PhysXAI/results/E1_multiseed_summary.json 2>/dev/null || \
 cat /kaggle/working/TC-PhysXAI/results/E1_summary.json 2>/dev/null || \
 echo 'Chưa có kết quả — chạy Section 2 trước.'

## 6. So sánh với XAI4Traj (E9)

So sánh TC-PhysXAI với baseline của thầy (XAI4Traj, Neurocomputing 2025) trên nhiều model, theo đúng giao thức R / P / H của paper (xem docstring của `E9_xai4traj_comparison.py`).

1. **6.1** liệt kê checkpoint đang có (kiểu model tự nhận từ `model_type`).
2. **6.2** chạy thử nhanh (3 storm, chỉ TC-XFlow) để kiểm tra pipeline.
3. **6.3** chạy đầy đủ: thêm `--model KIND:PATH` cho từng model khác (kind: `st_trans lstm gru rnn mmstn physdiff tcdiffuser mgtcf`) hoặc dùng `--auto THU_MUC_CHECKPOINT`.
4. **6.4** xem bảng kết quả.

E9 so 5 cách chia đoạn: `mdl rdp sliding` (của thầy), `phys_abc` (cố định theo thời gian) và `phys_regime` (**của bạn**: cắt tại điểm đổi chế độ chuyển động, theo hướng/tốc độ).

In [ ]:
# ── 6.1 Liệt kê checkpoint (tất cả dataset đã gắn)
!python /kaggle/working/TC-PhysXAI/Experiments/E9_xai4traj_comparison.py --scan /kaggle/input

In [ ]:
# ── 6.2 Chạy thử nhanh E9 (3 storm, chỉ TC-XFlow seed0)
!python /kaggle/working/TC-PhysXAI/Experiments/E9_xai4traj_comparison.py --quick \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results/E9_quick \
    --model fm:/kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth:TC-XFlow

In [ ]:
# ── 6.3 Chạy đầy đủ E9 — THAY các PATH_* bằng đường dẫn lấy được ở 6.1 (xoá dòng model nào bạn không có)
!python /kaggle/working/TC-PhysXAI/Experiments/E9_xai4traj_comparison.py \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results/E9 \
    --n_storms 20 --K 10 --n_perturb 32 --taus 10,25,50 \
    --model fm:/kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth:TC-XFlow \
    --model st_trans:PATH_ST_TRANS:ST-Trans \
    --model lstm:PATH_LSTM:LSTM \
    --model gru:PATH_GRU:GRU \
    --model mmstn:PATH_MMSTN:MMSTN \
    --model physdiff:PATH_PHYSDIFF:Phys-Diff \
    --model tcdiffuser:PATH_TCDIFFUSER:TC-Diffuser \
    --model mgtcf:PATH_MGTCF:MGTCF

In [ ]:
# ── 6.4 Xem kết quả (đổi tau25 -> tau10 / tau50 để xem ngưỡng khác)
!cat /kaggle/working/TC-PhysXAI/results/E9/E9_table_xai4traj_tau25.md
!echo
!cat /kaggle/working/TC-PhysXAI/results/E9/E9_ranking_summary_tau25.csv
!echo
!cat /kaggle/working/TC-PhysXAI/results/E9/E9_paired_bootstrap_tau25.csv

## 7. Giải thích theo đơn vị vật lý (E10) — đóng góp chính

Đơn vị giải thích = (đoạn chế độ chuyển động) × (TRACK hoặc nhóm ERA5 theo tầng/biến). Đo bằng thay đổi của **phân phối** dự báo (dọc/ngang quỹ đạo, độ phân tán, 24/48/72 h, energy distance), kiểm tra độ trung thực bằng đường xoá tích luỹ, và có **kiểm tra vật lý dòng dẫn hướng** (xoá gió U,V thì dự báo phải lệch ngược chiều dị thường gió).

**Chạy trên T4 — đọc trước:**
- Mỗi con số là nhiều lần chạy model. Sau storm đầu tiên log in `calls, giây, ETA`: nhân với số storm × số model để ước tính thời gian rồi mới chạy dài.
- Mọi cell đều có `--resume`: nếu session bị ngắt, chạy lại cell đó, model nào đã xong sẽ được bỏ qua (file `E10_part_*.csv`).
- Muốn nhẹ hơn: `--n_storms` nhỏ, `--modes neutral,swap`, `--scopes all,era5`, `--skip_greedy`, `--n_shapley_perm 0`.
- Pha (phase) của mỗi storm lấy từ **dự báo của chính model**, không dùng quỹ đạo thật.
- Model nào thực sự không dùng ERA5 sẽ cho tác động ERA5 = 0 (kết quả đúng, không phải lỗi). Lưu ý: LSTM/GRU/RNN trong repo vẫn nhận ERA5 qua FNO3D, nên không phải mốc "không dùng khí quyển".


In [ ]:
# ── 7.1 Chạy thử nhanh (3 storm, chỉ TC-XFlow) — xem ETA/lỗi trước khi chạy dài
!python /kaggle/working/TC-PhysXAI/Experiments/E10_joint_segments.py --quick \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results/E10_quick \
    --model fm:/kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth:TC-XFlow


In [ ]:
# ── 7.2 TC-XFlow, cấu hình đầy đủ (phase x level, 3 toán tử, Shapley, greedy)
!python /kaggle/working/TC-PhysXAI/Experiments/E10_joint_segments.py --resume \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results/E10 \
    --n_storms 20 --K 10 --segmentation phys_regime --era5_groups level \
    --model fm:/kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth:TC-XFlow


In [ ]:
# ── 7.3 Các model khác (nhẹ hơn) — THAY PATH_*, xoá dòng model không có.
#     Dùng CÙNG output_dir với 7.2 + --resume: TC-XFlow được nạp lại từ file part, bảng gộp tất cả model.
!python /kaggle/working/TC-PhysXAI/Experiments/E10_joint_segments.py --resume \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results/E10 \
    --n_storms 20 --K 10 --segmentation phys_regime --era5_groups level \
    --modes neutral,swap --scopes all,era5 --skip_greedy --n_shapley_perm 0 --n_random 2 \
    --model fm:/kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth:TC-XFlow \
    --model st_trans:PATH_ST_TRANS:ST-Trans \
    --model lstm:PATH_LSTM:LSTM \
    --model gru:PATH_GRU:GRU \
    --model mmstn:PATH_MMSTN:MMSTN \
    --model physdiff:PATH_PHYSDIFF:Phys-Diff \
    --model tcdiffuser:PATH_TCDIFFUSER:TC-Diffuser \
    --model mgtcf:PATH_MGTCF:MGTCF


In [ ]:
# ── 7.4 Độ nhạy với cách chia pha + lưới tầng x biến đầy đủ (TC-XFlow, 10 storm)
for seg in ['phys_abc', 'mdl']:
    !python /kaggle/working/TC-PhysXAI/Experiments/E10_joint_segments.py --resume \
        --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm --n_storms 10 --segmentation $seg \
        --output_dir /kaggle/working/TC-PhysXAI/results/E10_seg_$seg \
        --modes neutral,swap --scopes all,era5 --n_shapley_perm 0 \
        --model fm:/kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth:TC-XFlow
!python /kaggle/working/TC-PhysXAI/Experiments/E10_joint_segments.py --resume \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm --n_storms 10 --era5_groups grid \
    --output_dir /kaggle/working/TC-PhysXAI/results/E10_grid \
    --modes neutral,swap --scopes all,era5 --n_shapley_perm 4 \
    --model fm:/kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth:TC-XFlow


In [ ]:
# ── 7.5 Xem kết quả E10
R = '/kaggle/working/TC-PhysXAI/results/E10'
!cat $R/E10_aopc_disp.md
!echo; echo '--- paired bootstrap (A - B) ---'; cat $R/E10_paired_bootstrap.csv
!echo; echo '--- steering check ---'; cat $R/E10_steering_summary.csv
!echo; echo '--- reliance ---'; cat $R/E10_reliance_summary.csv
!echo; cat $R/E10_hypotheses.md
!echo; cat $R/E10_notes.md
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(R + '/E10_unit_heatmap_*.png')):
    display(Image(p))


---

## Appendix: ERA5 Channel Reference

| Index | Variable | Level |
|-------|----------|-------|
| 0–3 | GPH | 200/500/850/925 hPa |
| 4–7 | U-wind | 200/500/850/925 hPa |
| 8–11 | V-wind | 200/500/850/925 hPa |
| 12 | SST | Surface |

```bibtex
@article{nguyen2025tcxflow,
  title={TC-XFlow},
  author={Nguyen, Hong Gam and others},
  journal={Expert Systems with Applications}, year={2025}
}
```